# 07 — Análisis profundo por cluster (clima y cultivo)

Este notebook **no vuelve a clusterizar**: consume los clusters ya definidos en
`06_clustering_final.ipynb` (`OUTPUTS/06a_zonas_clusters.csv` y
`OUTPUTS/06b_perfiles_productivos_clusters.csv`) y profundiza en cada uno, más allá de
la visualización geográfica de puntos. Para cada método se muestra: medias por cluster
(clima y producción) y la evolución temporal 2020–2025 de esas métricas, con cultivos
representativos.

| Sección | Clusters que analiza | Unidad |
|---|---|---|
| A | Zonas agroclimáticas (06a) | 28 distritos / zona climática |
| B | Perfiles productivos (06b) | perfiles `(región, cultivo)` |

## Configuración común

In [1]:
# ====================================================================
# Importaciones y carga de datos + asignaciones de cluster ya calculadas
# ====================================================================
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif ROOT.name == "Clustering":
    ROOT = ROOT.parent

RUTA_DATA = ROOT / "OUTPUTS" / "dataset_integrado.csv"
RUTA_OUT = ROOT / "OUTPUTS"
RUTA_FIG = RUTA_OUT / "figures"
RUTA_FIG.mkdir(parents=True, exist_ok=True)

CLIMA_CORE = [
    "temp_promedio", "precipitacion", "humedad_relativa",
    "radiacion_solar", "humedad_suelo",
]
NOMBRES_CLIMA = {
    "temp_promedio": "Temp. promedio (°C)",
    "precipitacion": "Precipitación (mm/día)",
    "humedad_relativa": "Humedad relativa (%)",
    "radiacion_solar": "Radiación solar (MJ/m²/día)",
    "humedad_suelo": "Humedad suelo (índice)",
}


def titulo_centrado(texto, size=18):
    """Titulo de figura centrado y en negrita, mas prominente que los ejes."""
    return dict(text=f"<b>{texto}</b>", x=0.5, xanchor="center", font=dict(size=size))


df = pd.read_csv(RUTA_DATA)
zonas_cluster = pd.read_csv(RUTA_OUT / "06a_zonas_clusters.csv")[
    ["distrito", "region", "piso_ecologico", "cluster"]
].rename(columns={"cluster": "cluster_zona"})
perfiles_cluster = pd.read_csv(RUTA_OUT / "06b_perfiles_productivos_clusters.csv")[
    ["region", "cultivo", "cluster"]
].rename(columns={"cluster": "cluster_perfil"})

print(f"dataset_integrado: {df.shape[0]:,} filas")
print(f"Clusters zona (06a): {sorted(zonas_cluster['cluster_zona'].unique())}")
print(f"Clusters perfil (06b): {sorted(perfiles_cluster['cluster_perfil'].unique())}")

# ====================================================================
# Nombres descriptivos de cluster (validados contra cultivos/clima dominantes
# de 06a_zonas_clusters.csv y 06b_perfiles_productivos_clusters.csv) + paleta
# de color fija por cluster, reutilizada en todos los graficos de cada metodo.
# ====================================================================
NOMBRES_ZONA = {
    0: "Oasis Piura",
    1: "Sierra papera",
    2: "Selva SM-Junin",
    3: "Puna Puno",
    4: "Costa canera",
    5: "Costa uvera",
}
NOMBRES_PERFIL = {
    0: "Cierre de ano",
    1: "Invierno",
    2: "Todo el ano",
    3: "Ventana exportadora",
    4: "Todo o nada",
    5: "Papa atipica",
    6: "Otono andino",
}

PALETTE_ZONA = px.colors.qualitative.Set2
PALETTE_PERFIL = px.colors.qualitative.Set3
COLORS_ZONA = {f"C{c} = {NOMBRES_ZONA[c]}": PALETTE_ZONA[i % len(PALETTE_ZONA)]
               for i, c in enumerate(sorted(NOMBRES_ZONA))}
COLORS_PERFIL = {f"C{c} = {NOMBRES_PERFIL[c]}": PALETTE_PERFIL[i % len(PALETTE_PERFIL)]
                  for i, c in enumerate(sorted(NOMBRES_PERFIL))}
print("Nombres cluster zona:", NOMBRES_ZONA)
print("Nombres cluster perfil:", NOMBRES_PERFIL)

dataset_integrado: 2,376 filas
Clusters zona (06a): [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Clusters perfil (06b): [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Nombres cluster zona: {0: 'Oasis Piura', 1: 'Sierra papera', 2: 'Selva SM-Junin', 3: 'Puna Puno', 4: 'Costa canera', 5: 'Costa uvera'}
Nombres cluster perfil: {0: 'Cierre de ano', 1: 'Invierno', 2: 'Todo el ano', 3: 'Ventana exportadora', 4: 'Todo o nada', 5: 'Papa atipica', 6: 'Otono andino'}


## A. Zonas agroclimáticas — análisis por cluster de clima

Cada distrito hereda el cluster de su zona climática (`06a_zonas_clusters.csv`). Aquí se
mira **qué tan distinto es el clima y la producción entre clusters**, y cómo evolucionó
cada uno año a año (no solo el promedio 2020–2025 ya visto en `06_clustering_final`).

**Referencia de nombres (zona climática):**

| Cluster | Nombre | Cultivos / clima dominante | Riesgo principal |
|---|---|---|---|
| C0 | Oasis Piura | Arroz, plátano, mango, uva — valles irrigados (Chira/Piura) y bosque seco | Agua río arriba y El Niño |
| C1 | Sierra papera | Papa, alfalfa, avena forrajera — sierra de Junín y La Libertad (Mantaro y Huamachuco) | Heladas |
| C2 | Selva SM-Junín | Arroz, plátano, caña, palma aceitera, piña, naranja — selva de San Martín/Junín (incluye Jilili-Piura, que climáticamente se agrupa aquí pese a estar etiquetado como sierra) | Deforestación y cambios de cultivo |
| C3 | Puna Puno | Avena forrajera, alfalfa, papa — cultivos forrajeros que sostienen la ganadería de alpaca/vacuno >3,800m (inferido, el dataset no mide ganado directamente) | Heladas que arruinan campañas enteras |
| C4 | Costa cañera | Caña de azúcar (outlier de Virú), palta Hass — riego tecnificado | Sequía extrema |
| C5 | Costa uvera | Uva (cultivo #1), arroz, espárrago — exigen humedad constante | Falta de agua para riego continuo |

In [2]:
# ====================================================================
# Merge: cada fila del dataset hereda el cluster de su distrito/zona
# ====================================================================
df_a = df.merge(zonas_cluster[["distrito", "cluster_zona"]], on="distrito", how="left")
df_a["cluster_str"] = df_a["cluster_zona"].astype(int).map(lambda c: f"C{c} = {NOMBRES_ZONA[c]}")
orden_zona = sorted(df_a["cluster_str"].unique(), key=lambda s: int(s[1]))
print(f"Filas con cluster de zona asignado: {df_a['cluster_zona'].notna().sum():,} / {len(df_a):,}")

Filas con cluster de zona asignado: 2,376 / 2,376


### A.1 Medias por cluster (clima y producción)

In [3]:
# ====================================================================
# Tabla de medias climaticas y de produccion por cluster de zona
# ====================================================================
resumen_zona = df_a.groupby("cluster_str")[CLIMA_CORE + ["produccion_ton"]].mean().round(2).reindex(orden_zona)
resumen_zona.index.name = "cluster"
print(resumen_zona.to_string())

variables_clima = list(NOMBRES_CLIMA.values())
ncols = 3
nrows = -(-len(variables_clima) // ncols)
fig_bar_clima_zona = make_subplots(rows=nrows, cols=ncols, subplot_titles=variables_clima,
                                    horizontal_spacing=0.08, vertical_spacing=0.3)
for i, (var_key, var_nombre) in enumerate(NOMBRES_CLIMA.items()):
    row, col = i // ncols + 1, i % ncols + 1
    for cluster in orden_zona:
        fig_bar_clima_zona.add_trace(
            go.Bar(x=[cluster], y=[resumen_zona.loc[cluster, var_key]],
                   marker_color=COLORS_ZONA[cluster], name=cluster,
                   legendgroup=cluster, showlegend=(i == 0)),
            row=row, col=col,
        )
fig_bar_clima_zona.update_xaxes(showticklabels=False)
fig_bar_clima_zona.update_layout(
    title=titulo_centrado("Clima promedio por cluster de zona (2020-2025)"),
    width=1350, height=750, margin=dict(t=110, b=40),
    legend=dict(title="Cluster"),
)
fig_bar_clima_zona.show()

                     temp_promedio  precipitacion  humedad_relativa  radiacion_solar  humedad_suelo  produccion_ton
cluster                                                                                                            
C0 = Oasis Piura             25.58           0.49             61.88            19.62           0.41        31558.21
C1 = Sierra papera            8.89           0.97             73.23            18.27           0.47        22214.32
C2 = Selva SM-Junin          22.55           1.63             70.71            15.79           0.56        32421.32
C3 = Puna Puno                8.78           1.73             62.28            22.40           0.46       142152.79
C4 = Costa canera            19.89           0.26             72.21            19.92           0.34        73091.32
C5 = Costa uvera             19.70           0.59             60.88            21.64           0.40        21960.87


### A.2 Evolución temporal del clima por cluster

In [4]:
# ====================================================================
# Evolucion anual de variables climaticas clave, por cluster de zona
# ====================================================================
evol_clima_zona = df_a.groupby(["cluster_str", "anio"])[CLIMA_CORE].mean().reset_index()

fig_evol_clima_zona = make_subplots(rows=nrows, cols=ncols, subplot_titles=variables_clima,
                                     horizontal_spacing=0.08, vertical_spacing=0.3)
for i, (var_key, var_nombre) in enumerate(NOMBRES_CLIMA.items()):
    row, col = i // ncols + 1, i % ncols + 1
    for cluster in orden_zona:
        sub = evol_clima_zona[evol_clima_zona["cluster_str"] == cluster].sort_values("anio")
        fig_evol_clima_zona.add_trace(
            go.Scatter(x=sub["anio"], y=sub[var_key], mode="lines+markers", name=cluster,
                       legendgroup=cluster, showlegend=(i == 0), line=dict(color=COLORS_ZONA[cluster])),
            row=row, col=col,
        )
fig_evol_clima_zona.update_xaxes(title_text="Año", dtick=1, tickfont=dict(size=9))
fig_evol_clima_zona.update_layout(
    title=titulo_centrado("Evolución anual del clima por cluster de zona"),
    width=1350, height=780, margin=dict(t=110),
    legend=dict(title="Cluster"),
)
fig_evol_clima_zona.show()

### A.3 Evolución de la producción por cluster (con cultivos representativos)

In [5]:
# ====================================================================
# Evolucion anual de produccion total por cluster de zona
# ====================================================================
evol_prod_zona = df_a.groupby(["cluster_str", "anio"])["produccion_ton"].sum().reset_index()
fig_evol_prod_zona = px.line(
    evol_prod_zona, x="anio", y="produccion_ton", color="cluster_str", markers=True,
    category_orders={"cluster_str": orden_zona}, color_discrete_map=COLORS_ZONA,
    labels={"produccion_ton": "Producción total (t)", "cluster_str": "Cluster", "anio": "Año"},
)
fig_evol_prod_zona.update_xaxes(dtick=1)
fig_evol_prod_zona.update_layout(
    title=titulo_centrado("Evolución anual de producción por cluster de zona"),
    width=1100, height=520, margin=dict(t=90),
)
fig_evol_prod_zona.show()

In [6]:
# ====================================================================
# Cultivos representativos (top-3 por produccion total) y su evolucion anual
# ====================================================================
top_cultivos_zona = (df_a.groupby(["cluster_str", "cultivo"])["produccion_ton"].sum()
                     .reset_index().sort_values("produccion_ton", ascending=False)
                     .groupby("cluster_str").head(3))
cultivos_rep_zona = set(zip(top_cultivos_zona["cluster_str"], top_cultivos_zona["cultivo"]))

evol_cultivo_zona = (df_a.groupby(["cluster_str", "cultivo", "anio"])["produccion_ton"]
                     .sum().reset_index())
evol_cultivo_zona = evol_cultivo_zona[
    evol_cultivo_zona.apply(lambda r: (r["cluster_str"], r["cultivo"]) in cultivos_rep_zona, axis=1)
]

ncols2 = 3
nrows2 = -(-len(orden_zona) // ncols2)
PALETTE_CULTIVO = px.colors.qualitative.Dark24
fig_evol_cultivo_zona = make_subplots(rows=nrows2, cols=ncols2, subplot_titles=orden_zona,
                                       horizontal_spacing=0.07, vertical_spacing=0.32)
vistos_cultivo_zona = set()
for i, cluster in enumerate(orden_zona):
    row, col = i // ncols2 + 1, i % ncols2 + 1
    sub_cluster = evol_cultivo_zona[evol_cultivo_zona["cluster_str"] == cluster]
    for j, cultivo in enumerate(sorted(sub_cluster["cultivo"].unique())):
        sub = sub_cluster[sub_cluster["cultivo"] == cultivo].sort_values("anio")
        fig_evol_cultivo_zona.add_trace(
            go.Scatter(x=sub["anio"], y=sub["produccion_ton"], mode="lines+markers", name=cultivo,
                       legendgroup=cultivo, showlegend=(cultivo not in vistos_cultivo_zona),
                       line=dict(color=PALETTE_CULTIVO[j % len(PALETTE_CULTIVO)], width=2)),
            row=row, col=col,
        )
        vistos_cultivo_zona.add(cultivo)
fig_evol_cultivo_zona.update_xaxes(title_text="Año", dtick=1, tickfont=dict(size=9))
fig_evol_cultivo_zona.update_yaxes(title_text="Producción (t)", exponentformat="SI", tickfont=dict(size=9))
fig_evol_cultivo_zona.update_layout(
    title=titulo_centrado("Evolución anual de los cultivos dominantes por cluster de zona"),
    width=1550, height=1000, margin=dict(t=110),
    legend=dict(title="Cultivo"),
)
fig_evol_cultivo_zona.write_image(str(RUTA_FIG / "07a_evolucion_cultivos_zona.png"), scale=2,
                                   width=1550, height=1000)
fig_evol_cultivo_zona.show()

## B. Perfiles productivos — análisis por cluster de cultivo

Cada combinación `(región, cultivo)` hereda el cluster de su perfil productivo
(`06b_perfiles_productivos_clusters.csv`). Aquí se mira la **producción y el clima
asociado por cluster**, y la evolución anual de los cultivos que lo componen.

In [7]:
# ====================================================================
# Merge: cada fila del dataset hereda el cluster de su perfil (region, cultivo)
# ====================================================================
df_b = df.merge(perfiles_cluster[["region", "cultivo", "cluster_perfil"]],
                 on=["region", "cultivo"], how="left")
df_b["cluster_str"] = df_b["cluster_perfil"].astype(int).map(lambda c: f"C{c} = {NOMBRES_PERFIL[c]}")
orden_perfil = sorted(df_b["cluster_str"].unique(), key=lambda s: int(s[1]))
print(f"Filas con cluster de perfil asignado: {df_b['cluster_perfil'].notna().sum():,} / {len(df_b):,}")

Filas con cluster de perfil asignado: 2,376 / 2,376


**Referencia de nombres (perfil productivo):**

| Cluster | Nombre | Cultivos | Pico | Nota |
|---|---|---|---|---|
| C0 | Cierre de año | Tomate (Ica), uva (Piura) | Nov-Dic | Concentración de fin de año |
| C1 | Invierno | Cítricos, palta, café, papa, arroz, maíz amarillo, avena forrajera | May-Jul | Cosecha de invierno andino-costero |
| C2 | Todo el año | Yuca, plátano, palma, caña, alfalfa, espárrago, naranja, piña, arroz/maíz (San Martín) | Sin pico (12 meses) | Perennes o riego tecnificado, cosecha continua |
| C3 | Ventana exportadora | Uva (Ica), mango (Piura) | Enero | Estrategia comercial: abastece Hemisferio Norte en su invierno |
| C4 | Todo o nada | Arroz (La Libertad), avena y papa (Puno) | Abril-Mayo | Un solo ciclo fuerte; sin plan B si falla |
| C5 | Papa atípica | Papa (Ica) | Agosto | Único perfil costero de papa; cluster de un solo elemento |
| C6 | Otoño andino | Maíz choclo (Junín), alfalfa (Puno) | Abril | Cierre de temporada de lluvias andina |

### B.1 Medias por cluster (producción y clima asociado)

In [8]:
# ====================================================================
# Tabla de medias productivas y climaticas por cluster de perfil
# ====================================================================
resumen_perfil = df_b.groupby("cluster_str").agg(
    produccion_media_ton=("produccion_ton", "mean"),
    produccion_total_ton=("produccion_ton", "sum"),
    n_perfiles=("cultivo", "nunique"),
    **{f"{v}_mean": (v, "mean") for v in CLIMA_CORE},
).round(2).reindex(orden_perfil)
print(resumen_perfil.to_string())

# Barras horizontales (en vez de leyenda + ejes ocultos): el nombre del cluster
# queda escrito junto a su barra, sin inclinar y sin necesidad de leyenda aparte.
orden_perfil_inv = orden_perfil[::-1]  # para que C0 quede arriba al graficar
fig_bar_clima_perfil = make_subplots(rows=nrows, cols=ncols, subplot_titles=variables_clima,
                                      horizontal_spacing=0.18, vertical_spacing=0.3)
for i, (var_key, var_nombre) in enumerate(NOMBRES_CLIMA.items()):
    row, col = i // ncols + 1, i % ncols + 1
    valores = [resumen_perfil.loc[c, f"{var_key}_mean"] for c in orden_perfil_inv]
    fig_bar_clima_perfil.add_trace(
        go.Bar(y=orden_perfil_inv, x=valores, orientation="h",
               marker_color=[COLORS_PERFIL[c] for c in orden_perfil_inv], showlegend=False),
        row=row, col=col,
    )
fig_bar_clima_perfil.update_yaxes(tickfont=dict(size=10))
fig_bar_clima_perfil.update_layout(
    title=titulo_centrado("Clima promedio asociado a cada cluster de perfil productivo"),
    width=1550, height=850, margin=dict(t=110, l=20),
)
fig_bar_clima_perfil.show()

                          produccion_media_ton  produccion_total_ton  n_perfiles  temp_promedio_mean  precipitacion_mean  humedad_relativa_mean  radiacion_solar_mean  humedad_suelo_mean
cluster_str                                                                                                                                                                              
C0 = Cierre de ano                    16053.27            2167191.42           2               22.30                0.50                  60.18                 21.02                0.40
C1 = Invierno                         22653.87           13456399.83           7               17.03                0.56                  71.56                 18.66                0.41
C2 = Todo el ano                      57608.24           54670218.31          10               21.24                1.28                  69.69                 17.25                0.51
C3 = Ventana exportadora              34200.17            4514422.83  

### B.2 Evolución anual de producción por cultivo dentro de cada cluster

In [9]:
# ====================================================================
# Evolucion anual de produccion por cultivo, agrupado por cluster de perfil
# ====================================================================
evol_perfil = (df_b.groupby(["cluster_str", "region", "cultivo", "anio"])["produccion_ton"]
              .sum().reset_index())
evol_perfil["etiqueta"] = evol_perfil["region"] + " - " + evol_perfil["cultivo"]

# 3x3 (en vez de 4x2): paneles mas grandes para que las lineas no se vean apiladas.
ncols3, nrows3 = 3, 3
PALETTE_REGION_CULTIVO = px.colors.qualitative.Dark24
fig_evol_perfil = make_subplots(
    rows=nrows3, cols=ncols3,
    subplot_titles=orden_perfil + [""] * (nrows3 * ncols3 - len(orden_perfil)),
    horizontal_spacing=0.06, vertical_spacing=0.22,
)
vistos_perfil = set()
for i, cluster in enumerate(orden_perfil):
    row, col = i // ncols3 + 1, i % ncols3 + 1
    sub_cluster = evol_perfil[evol_perfil["cluster_str"] == cluster]
    for j, etiqueta in enumerate(sorted(sub_cluster["etiqueta"].unique())):
        sub = sub_cluster[sub_cluster["etiqueta"] == etiqueta].sort_values("anio")
        fig_evol_perfil.add_trace(
            go.Scatter(x=sub["anio"], y=sub["produccion_ton"], mode="lines+markers", name=etiqueta,
                       legendgroup=etiqueta, showlegend=(etiqueta not in vistos_perfil),
                       line=dict(color=PALETTE_REGION_CULTIVO[j % len(PALETTE_REGION_CULTIVO)], width=2),
                       marker=dict(size=5)),
            row=row, col=col,
        )
        vistos_perfil.add(etiqueta)
fig_evol_perfil.update_xaxes(title_text="Año", dtick=1, tickfont=dict(size=9))
fig_evol_perfil.update_yaxes(title_text="Producción (t)", exponentformat="SI", tickfont=dict(size=9))
fig_evol_perfil.update_layout(
    title=titulo_centrado("Evolución anual de producción por cultivo, dentro de cada cluster de perfil"),
    width=1700, height=1350, margin=dict(t=110),
    legend=dict(title="Región - cultivo", font=dict(size=9)),
)
fig_evol_perfil.write_image(str(RUTA_FIG / "07b_evolucion_cultivos_perfil.png"), scale=2,
                             width=1700, height=1350)
fig_evol_perfil.show()

### B.3 Producción total por cluster (escala log, con outliers)

In [10]:
# ====================================================================
# Distribucion de produccion total (suma 2020-2025) por perfil, dentro de cada cluster
# ====================================================================
total_por_perfil = (df_b.groupby(["cluster_str", "region", "cultivo"])["produccion_ton"]
                    .sum().reset_index())
total_por_perfil = total_por_perfil[total_por_perfil["produccion_ton"] > 0]

# Mismo estilo simple que el boxplot de zona en 06_clustering_final.ipynb: un color
# solido (sin colorear por cluster ni leyenda, esa info ya esta en el eje x).
fig_box_perfil = px.box(
    total_por_perfil, x="cluster_str", y="produccion_ton", log_y=True, points="outliers",
    category_orders={"cluster_str": orden_perfil},
    labels={"produccion_ton": "Producción total 2020-2025 (t, log)", "cluster_str": "Cluster"},
)
fig_box_perfil.update_traces(marker_color="#4C78A8")
# exponentformat="SI" evita mezclar numeros planos (2000000) con sufijos (10M) en el mismo eje
fig_box_perfil.update_yaxes(exponentformat="SI", dtick=1)
fig_box_perfil.update_xaxes(tickangle=0, tickfont=dict(size=10))
fig_box_perfil.update_layout(
    title=titulo_centrado("Distribución de producción total por cultivo, dentro de cada cluster de perfil"),
    width=1500, height=600, margin=dict(b=80, t=100),
)
fig_box_perfil.show()

## Conclusión

Este notebook complementa a `06_clustering_final.ipynb`: ese define y valida los
clusters (Silhouette, ARI, mapas); este profundiza en **qué hay dentro** de cada uno
(medias, evolución anual, cultivos representativos). Para la definición metodológica de
cada cluster y por qué se eligió cada enfoque, ver `06_clustering_final.ipynb` y
`BORRADORES/06_clustering_cultivos.ipynb` (método descartado, con la razón documentada).